In [12]:
from pathlib import Path
from joblib import dump

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

In [13]:
RANDOM_STATE = 22
Z_SCORE_THRESHOLD = 3.0
ARTIFACTS_PATH = Path(r"C:\Users\Youcode\Documents\DEV_AI\sprint_1\soutnance_1\artifacts")

### Preprocessing



In [14]:
path = Path("data/raw/student-mental.csv")
if not path.is_file():
    path = Path("..") / path

df = pd.read_csv(path)
target = "Mental_Health_Risk"
X = df.drop(columns=["Student_ID", target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [15]:
ordinal_categories = {
    "Year_of_Study": [1, 2, 3, 4, 5],
    "Physical_Activity": ["Low", "Moderate", "High"],
    "Diet_Quality": ["Poor", "Average", "Good"],
}
ordinal_columns = list(ordinal_categories)
nominal_columns = ["Gender", "Part_Time_Job", "Previous_Mental_Health_History"]
numerical_columns = X_train.drop(columns=ordinal_columns).select_dtypes(include="number").columns

### Z-score outlier treatment

In [16]:
def z_score(column, dataframe):
    values = dataframe[column]
    mean = values.mean()
    std = values.std(ddof=0)
    if std == 0:
        std = 1

    result = pd.DataFrame({column: values})
    result["z_score"] = (values - mean) / std
    result["is_outlier"] = result["z_score"].abs() > Z_SCORE_THRESHOLD
    result["capped_z_score"] = result["z_score"].clip(-Z_SCORE_THRESHOLD, Z_SCORE_THRESHOLD)
    return result

#### Apply the rule to the student dataset

In [17]:
for column in numerical_columns:
    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = X_train[column].dropna()
    outliers = outliers[
        (outliers < lower) | (outliers > upper)
    ]
    z_score_result = z_score(column, X_train)
    z_score_count = z_score_result["is_outlier"].sum()

    print(f"{column}: {len(outliers)} IQR flags, {z_score_count} z-score flags")

Age: 3 IQR flags, 3 z-score flags
Sleep_Hours: 8 IQR flags, 0 z-score flags
Study_Hours_Per_Week: 5 IQR flags, 3 z-score flags
GPA: 7 IQR flags, 4 z-score flags
Social_Media_Hours: 19 IQR flags, 5 z-score flags
Screen_Time_Hours: 11 IQR flags, 4 z-score flags
Financial_Stress: 0 IQR flags, 0 z-score flags
Family_Support: 0 IQR flags, 0 z-score flags
Attendance_Rate: 7 IQR flags, 6 z-score flags


In [18]:
numerical_transformer = make_pipeline(
    StandardScaler(),
    FunctionTransformer(
        np.clip,
        kw_args={"a_min": -Z_SCORE_THRESHOLD, "a_max": Z_SCORE_THRESHOLD},
        feature_names_out="one-to-one",
    ),
    SimpleImputer(strategy="median"),
)
nominal_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)
ordinal_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OrdinalEncoder(
        categories=list(ordinal_categories.values()),
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    ),
)

In [19]:
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numerical_transformer, numerical_columns),
    ("ordinal", ordinal_transformer, ordinal_columns),
    ("nominal", nominal_transformer, nominal_columns),
])

In [20]:
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

### save the preprocessing pipline and splits


In [21]:
X_train.to_csv(ARTIFACTS_PATH / "X_train.csv")
X_test.to_csv(ARTIFACTS_PATH / "x_test.csv")
y_train.to_csv(ARTIFACTS_PATH / "y_train.csv")
y_test.to_csv(ARTIFACTS_PATH / "y_test.csv")

In [22]:
pipline = {
    "preprocessor": clone(preprocessor),
    "X_train": X_train,
    "y_train": y_train,
    "target": target,
}
dump(pipline, Path(ARTIFACTS_PATH) /"preprocessing.joblib")

['C:\\Users\\Youcode\\Documents\\DEV_AI\\sprint_1\\soutnance_1\\artifacts\\preprocessing.joblib']